# Model Eval — Raw Output Inspector

In [1]:
import json
import sys
import time
from pathlib import Path

import ipywidgets as widgets
from dotenv import load_dotenv
from IPython.display import display, JSON, HTML

sys.path.insert(0, '../src')
load_dotenv('../.env')

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.tasks import run_ner, run_summary, run_tags

EVAL_DOCS_DIR = Path('../data/model_eval_docs')
RESULTS_PATH  = Path('../data/results/model_eval.json')

config = load_config('../config.toml')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')
BACKEND = config['run']['backend']

with open(RESULTS_PATH) as f:
    records = json.load(f)

index = {}
for r in records:
    index[(r['model'], r['doc'], r['task'])] = r

models = sorted(set(r['model'] for r in records))
docs   = sorted(set(r['doc']   for r in records))
tasks  = sorted(set(r['task']  for r in records))

TASK_FNS = {'ner': run_ner, 'tags': run_tags, 'summary': run_summary}

print(f'backend={BACKEND}  {len(records)} records  {len(models)} models  {len(docs)} docs')

backend=ollama  567 records  27 models  7 docs


In [2]:
w_model  = widgets.Select(options=models, description='Model', rows=10, layout=widgets.Layout(width='320px'))
w_doc    = widgets.Select(options=docs,   description='Doc',   rows=10, layout=widgets.Layout(width='600px'))
w_task   = widgets.ToggleButtons(options=tasks, description='Task')
w_rerun  = widgets.Button(description='Rerun', button_style='warning', icon='refresh')
w_save   = widgets.Checkbox(value=False, description='Save result to model_eval.json', indent=False)
out      = widgets.Output()


def _show_record(record):
    elapsed = record.get('elapsed_s')
    error   = record.get('error')
    result  = record.get('result')

    parts = []
    if elapsed is not None:
        parts.append(f'<b>elapsed:</b> {elapsed:.1f} s')
    if error:
        parts.append(f'<b style="color:crimson">error:</b> {error}')
    if parts:
        display(HTML(' &nbsp;&nbsp; '.join(parts)))

    if result is None:
        display(HTML('<p style="color:grey;font-style:italic">No output (empty response).</p>'))
    elif isinstance(result, (dict, list)):
        display(JSON(result))
    else:
        display(HTML(f'<pre style="white-space:pre-wrap">{result}</pre>'))


def _render(_change=None):
    key = (w_model.value, w_doc.value, w_task.value)
    record = index.get(key)
    with out:
        out.clear_output()
        if record is None:
            display(HTML('<p style="color:grey">No record found for this combination.</p>'))
            return
        display(HTML('<i style="color:grey">Cached result</i>'))
        _show_record(record)


def _rerun(_click=None):
    model = w_model.value
    doc   = w_doc.value
    task  = w_task.value

    with out:
        out.clear_output()
        display(HTML(f'<i>Running <b>{task}</b> on <b>{model}</b> / {doc} ...</i>'))

    cfg = {k: (v.copy() if isinstance(v, dict) else v) for k, v in config.items()}
    cfg[BACKEND] = config[BACKEND].copy()
    cfg[BACKEND]['model'] = model

    post = load_note(str(EVAL_DOCS_DIR / doc))
    t0 = time.perf_counter()
    error = None
    result = None
    try:
        result = TASK_FNS[task](BACKEND, post.content, cfg)
    except Exception as e:
        error = f'ERROR: {e}'
    elapsed = round(time.perf_counter() - t0, 1)

    new_record = {'model': model, 'task': task, 'doc': doc,
                  'elapsed_s': elapsed, 'result': result, 'error': error}

    # Update in-memory index
    key = (model, doc, task)
    index[key] = new_record
    for i, r in enumerate(records):
        if (r['model'], r['doc'], r['task']) == key:
            records[i] = new_record
            break
    else:
        records.append(new_record)

    if w_save.value:
        RESULTS_PATH.write_text(json.dumps(records, ensure_ascii=False, indent=2))

    with out:
        out.clear_output()
        label = 'Rerun result' + (' (saved)' if w_save.value else ' (not saved)')
        display(HTML(f'<i style="color:grey">{label}</i>'))
        _show_record(new_record)


w_model.observe(_render, names='value')
w_doc.observe(_render,   names='value')
w_task.observe(_render,  names='value')
w_rerun.on_click(_rerun)

display(
    widgets.VBox([
        widgets.HBox([w_model, w_doc]),
        widgets.HBox([w_task, w_rerun, w_save]),
        out,
    ])
)
_render()